In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE CATALOG crypto_pipeline")

DataFrame[]

In [0]:
# Databricks notebook source

# ==========================================================
# GOLD LAYER
#
# Notebook:
# 02_gold_rolling_metrics
#
# Purpose:
# Calculate rolling metrics
#
# Grain:
# One row per Coin per Observation Timestamp
#
# ==========================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "crypto_pipeline"

FACT_TABLE = f"{CATALOG}.silver.fact_coin_price"
GOLD_TABLE = f"{CATALOG}.gold.rolling_metrics"

spark.sql(f"USE CATALOG {CATALOG}")

# ==========================================================
# Create Gold Table
# ==========================================================

spark.sql(f"""

CREATE TABLE IF NOT EXISTS {GOLD_TABLE}

(

coin_sk STRING,

coin_id STRING,

observation_ts TIMESTAMP,

rolling_avg_price_7 DOUBLE,

rolling_avg_price_30 DOUBLE,

rolling_avg_volume_7 DOUBLE,

price_volatility_7 DOUBLE

)

USING DELTA

""")

# ==========================================================
# Read Fact Table
# ==========================================================

fact_df = spark.table(FACT_TABLE)

display(fact_df)

coin_sk,coin_id,observation_ts,current_price,market_cap,total_volume,market_cap_rank,price_change_24h,price_change_percentage_24h
2e718161445afbd9a5ee5c5a22835a07defc2037fe5031de1fef450dd4aac94b,aster-2,2026-07-22T08:58:57.519Z,0.622088,1.670620127E9,6.3981992E7,46,-0.00839127419145902,-1.33094
2e718161445afbd9a5ee5c5a22835a07defc2037fe5031de1fef450dd4aac94b,aster-2,2026-07-22T08:59:21.134Z,0.622088,1.670620127E9,6.3981992E7,46,-0.00839127419145902,-1.33094
4e2b4465ae2c4c917495b637780dff7ec2113e1f390b8aa84fbb19819b848768,avalanche-2,2026-07-22T08:58:57.519Z,6.51,2.812036807E9,1.21168985E8,32,-0.14037606851970175,-2.10982
4e2b4465ae2c4c917495b637780dff7ec2113e1f390b8aa84fbb19819b848768,avalanche-2,2026-07-22T08:59:21.134Z,6.51,2.812036807E9,1.21168985E8,32,-0.14037606851970175,-2.10982
4be6b588bed71a3f34047d227bfad49e9f35d4a89195b7ad153f865a9450796a,binancecoin,2026-07-22T08:58:57.519Z,569.71,7.587212412E10,5.64901171E8,4,-7.535549706710867,-1.30542
4be6b588bed71a3f34047d227bfad49e9f35d4a89195b7ad153f865a9450796a,binancecoin,2026-07-22T08:59:21.134Z,569.71,7.587212412E10,5.64901171E8,4,-7.535549706710867,-1.30542
c18cad259c9f4f8ef98ef645b2aa3dac7d726ab041d06643d5d13ffff58d358c,bitcoin,2026-07-22T08:58:57.519Z,65922.0,1.322409077378E12,3.2062806671E10,1,-251.1924731802137,-0.3796
c18cad259c9f4f8ef98ef645b2aa3dac7d726ab041d06643d5d13ffff58d358c,bitcoin,2026-07-22T08:59:21.134Z,65922.0,1.322409077378E12,3.2062806671E10,1,-251.1924731802137,-0.3796
c4e03185c2e365e4f3bb253b58944a0252139e6cb7d6ecae7a7d2dde8ab47ef0,bitcoin-cash,2026-07-22T08:58:57.519Z,220.78,4.430395854E9,8.4996954E7,23,-2.645743369264551,-1.1842
c4e03185c2e365e4f3bb253b58944a0252139e6cb7d6ecae7a7d2dde8ab47ef0,bitcoin-cash,2026-07-22T08:59:21.134Z,220.78,4.430395854E9,8.4996954E7,23,-2.645743369264551,-1.1842


In [0]:
window7 = (

    Window

    .partitionBy("coin_sk")

    .orderBy("observation_ts")

    .rowsBetween(-6, 0)

)

window30 = (

    Window

    .partitionBy("coin_sk")

    .orderBy("observation_ts")

    .rowsBetween(-29, 0)

)

In [0]:
rolling_df = (

    fact_df

    .withColumn(

        "rolling_avg_price_7",

        F.avg(

            "current_price"

        ).over(window7)

    )

    .withColumn(

        "rolling_avg_price_30",

        F.avg(

            "current_price"

        ).over(window30)

    )

    .withColumn(

        "rolling_avg_volume_7",

        F.avg(

            "total_volume"

        ).over(window7)

    )

    .withColumn(

        "price_volatility_7",

        F.stddev(

            "current_price"

        ).over(window7)

    )

)

display(rolling_df)

coin_sk,coin_id,observation_ts,current_price,market_cap,total_volume,market_cap_rank,price_change_24h,price_change_percentage_24h,rolling_avg_price_7,rolling_avg_price_30,rolling_avg_volume_7,price_volatility_7
124d4b609018175dd91ff6a12b372c389f9b0046298e32353f098c0b2b2995ea,usds,2026-07-22T08:58:57.519Z,0.999865,9.910984377E9,1.11249065E8,13,2.44E-5,0.00244,0.999865,0.999865,1.11249065E8,null
124d4b609018175dd91ff6a12b372c389f9b0046298e32353f098c0b2b2995ea,usds,2026-07-22T08:59:21.134Z,0.999865,9.910984377E9,1.11249065E8,13,2.44E-5,0.00244,0.999865,0.999865,1.11249065E8,0.0
126bda23e48f287091cc02ebdc4a972096a6bc774223da74dc40c1ab85f18249,canton-network,2026-07-22T08:58:57.519Z,0.124072,4.853486737E9,8252504.0,21,-0.001422113417409582,-1.13321,0.124072,0.124072,8252504.0,null
126bda23e48f287091cc02ebdc4a972096a6bc774223da74dc40c1ab85f18249,canton-network,2026-07-22T08:59:21.134Z,0.124072,4.853486737E9,8252504.0,21,-0.001422113417409582,-1.13321,0.124072,0.124072,8252504.0,0.0
15693b928f9f1c8df421b733df7d6adbcafded5444e309cecdac7a7510520edc,ripple,2026-07-22T08:58:57.519Z,1.13,7.0829003636E10,1.35471518E9,6,9.2615E-4,0.08175,1.13,1.13,1.35471518E9,null
15693b928f9f1c8df421b733df7d6adbcafded5444e309cecdac7a7510520edc,ripple,2026-07-22T08:59:21.134Z,1.13,7.0829003636E10,1.35471518E9,6,9.2615E-4,0.08175,1.13,1.13,1.35471518E9,0.0
20ef3a64f532468d1178902f57cc6e910c7d608e3c7821f8eb4450295adccaa2,htx-dao,2026-07-22T08:58:57.519Z,1.84E-6,1.651554307E9,4.4443701E7,47,-1.3094524611E-8,-0.70714,1.84E-6,1.84E-6,4.4443701E7,null
20ef3a64f532468d1178902f57cc6e910c7d608e3c7821f8eb4450295adccaa2,htx-dao,2026-07-22T08:59:21.134Z,1.84E-6,1.651554307E9,4.4443701E7,47,-1.3094524611E-8,-0.70714,1.84E-6,1.84E-6,4.4443701E7,0.0
23fec56b98d88017bcab27610360b3ed3e1ad86e04aed9093989d62cb6090b03,near,2026-07-22T08:58:57.519Z,1.88,2.44767747E9,2.02695092E8,38,-0.1251788552169344,-6.24059,1.88,1.88,2.02695092E8,null
23fec56b98d88017bcab27610360b3ed3e1ad86e04aed9093989d62cb6090b03,near,2026-07-22T08:59:21.134Z,1.88,2.44767747E9,2.02695092E8,38,-0.1251788552169344,-6.24059,1.88,1.88,2.02695092E8,0.0


In [0]:
rolling_stage = (

    rolling_df

    .select(

        "coin_sk",

        "coin_id",

        "observation_ts",

        "rolling_avg_price_7",

        "rolling_avg_price_30",

        "rolling_avg_volume_7",

        "price_volatility_7"

    )

)

display(rolling_stage)

print(

    "Rows:",

    rolling_stage.count()

)

coin_sk,coin_id,observation_ts,rolling_avg_price_7,rolling_avg_price_30,rolling_avg_volume_7,price_volatility_7
124d4b609018175dd91ff6a12b372c389f9b0046298e32353f098c0b2b2995ea,usds,2026-07-22T08:58:57.519Z,0.999865,0.999865,1.11249065E8,null
124d4b609018175dd91ff6a12b372c389f9b0046298e32353f098c0b2b2995ea,usds,2026-07-22T08:59:21.134Z,0.999865,0.999865,1.11249065E8,0.0
126bda23e48f287091cc02ebdc4a972096a6bc774223da74dc40c1ab85f18249,canton-network,2026-07-22T08:58:57.519Z,0.124072,0.124072,8252504.0,null
126bda23e48f287091cc02ebdc4a972096a6bc774223da74dc40c1ab85f18249,canton-network,2026-07-22T08:59:21.134Z,0.124072,0.124072,8252504.0,0.0
15693b928f9f1c8df421b733df7d6adbcafded5444e309cecdac7a7510520edc,ripple,2026-07-22T08:58:57.519Z,1.13,1.13,1.35471518E9,null
15693b928f9f1c8df421b733df7d6adbcafded5444e309cecdac7a7510520edc,ripple,2026-07-22T08:59:21.134Z,1.13,1.13,1.35471518E9,0.0
20ef3a64f532468d1178902f57cc6e910c7d608e3c7821f8eb4450295adccaa2,htx-dao,2026-07-22T08:58:57.519Z,1.84E-6,1.84E-6,4.4443701E7,null
20ef3a64f532468d1178902f57cc6e910c7d608e3c7821f8eb4450295adccaa2,htx-dao,2026-07-22T08:59:21.134Z,1.84E-6,1.84E-6,4.4443701E7,0.0
23fec56b98d88017bcab27610360b3ed3e1ad86e04aed9093989d62cb6090b03,near,2026-07-22T08:58:57.519Z,1.88,1.88,2.02695092E8,null
23fec56b98d88017bcab27610360b3ed3e1ad86e04aed9093989d62cb6090b03,near,2026-07-22T08:59:21.134Z,1.88,1.88,2.02695092E8,0.0


Rows: 100


In [0]:
from delta.tables import DeltaTable

gold_delta = DeltaTable.forName(

    spark,

    GOLD_TABLE

)

(

    gold_delta.alias("target")

    .merge(

        rolling_stage.alias("source"),

        """

        target.coin_sk = source.coin_sk

        AND

        target.observation_ts = source.observation_ts

        """

    )

    .whenMatchedUpdate(

        set={

            "rolling_avg_price_7":

                "source.rolling_avg_price_7",

            "rolling_avg_price_30":

                "source.rolling_avg_price_30",

            "rolling_avg_volume_7":

                "source.rolling_avg_volume_7",

            "price_volatility_7":

                "source.price_volatility_7"

        }

    )

    .whenNotMatchedInsert(

        values={

            "coin_sk":"source.coin_sk",

            "coin_id":"source.coin_id",

            "observation_ts":"source.observation_ts",

            "rolling_avg_price_7":

                "source.rolling_avg_price_7",

            "rolling_avg_price_30":

                "source.rolling_avg_price_30",

            "rolling_avg_volume_7":

                "source.rolling_avg_volume_7",

            "price_volatility_7":

                "source.price_volatility_7"

        }

    )

    .execute()

)

print("Rolling Metrics Merge Completed")

Rolling Metrics Merge Completed


In [0]:
gold_df = spark.table(GOLD_TABLE)

display(

    gold_df.orderBy(

        F.desc("observation_ts")

    )

)

print(

    "Rows:",

    gold_df.count()

)

coin_sk,coin_id,observation_ts,rolling_avg_price_7,rolling_avg_price_30,rolling_avg_volume_7,price_volatility_7
4e2b4465ae2c4c917495b637780dff7ec2113e1f390b8aa84fbb19819b848768,avalanche-2,2026-07-22T08:59:21.134Z,6.51,6.51,1.21168985E8,0.0
2566e75a9ea09ae330ae3bb7a9417a3fcdbe5696fb322ab37f3c3d24c3004c76,pax-gold,2026-07-22T08:59:21.134Z,4106.34,4106.34,1.2553983E8,0.0
55c4b7c193a181cfb8a4e4e4f1204d68f98c0fbdaf391148674669f309781930,leo-token,2026-07-22T08:59:21.134Z,9.72,9.72,176846.0,0.0
95b12ebb7f9edf3d7b5a9a3faef2e33aa0db75684755729b1d7938c11214fc6c,figure-heloc,2026-07-22T08:59:21.134Z,1.005,1.005,5.5088452E7,0.0
4a58a2c0e14acb98c656a990067f07e775e933535be0563e61def03cb586fa96,zcash,2026-07-22T08:59:21.134Z,516.48,516.48,3.34635145E8,0.0
69e854aaab9c9b25ed1fa04ab61957db9ac4c12488b0620286525a851a65dcf6,stellar,2026-07-22T08:59:21.134Z,0.189427,0.189427,1.69666248E8,0.0
6235bef37881100ed67526ae2a1af07ab4725e2e7b1f7ae709c29e01a6bc53e2,cardano,2026-07-22T08:59:21.134Z,0.172065,0.172065,2.82280937E8,0.0
e2397c39195878c724b52c41f7b5e9587bb0e7a6e4fdbc3d2cc4e1f7aaefb3ba,tron,2026-07-22T08:59:21.134Z,0.328866,0.328866,3.25557615E8,0.0
c00314f175383d608c77296a3493d91cdb2c1ec381d1e8f345ce9c447064d2e1,the-open-network,2026-07-22T08:59:21.134Z,1.51,1.51,7.3056643E7,0.0
20ef3a64f532468d1178902f57cc6e910c7d608e3c7821f8eb4450295adccaa2,htx-dao,2026-07-22T08:59:21.134Z,1.84E-6,1.84E-6,4.4443701E7,0.0


Rows: 100
